In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Delta_Project").getOrCreate()


In [0]:
data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]


In [0]:
columns = ["id", "customer_id", "product", "amount"]


In [0]:
df = spark.createDataFrame(data, columns)

In [0]:
df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/detlass")


In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/detlass")
df.show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  1|       C001| Laptop| 50000|
|  2|       C002| Mobile| 15000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
+---+-----------+-------+------+



In [0]:
new_data = [(5, "C005", "Camera", 30000)]


new_df = spark.createDataFrame(new_data, columns)

new_df.write.format("delta").mode("append").save("/Volumes/workspace/default/detlass")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/Volumes/workspace/default/detlass")

delta_table.update(
    condition="id = 2",
    set={"amount": "18000"}
)

DataFrame[num_affected_rows: bigint]

In [0]:
delta_table.delete("id = 1")

DataFrame[num_affected_rows: bigint]

In [0]:
updates = [
    (3, "C003", "Tablet", 22000),
    (6, "C006", "Watch", 8000)
]

updates_df = spark.createDataFrame(updates, columns)

delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
from pyspark.sql.functions import lit

df = spark.read.format("delta").load("/Volumes/workspace/default/detlass")

df = df.withColumn("country", lit("India"))

df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save("/Volumes/workspace/default/detlass")

In [0]:
spark.sql("""
DESCRIBE HISTORY delta.`/Volumes/workspace/default/detlass`
""").show(truncate=False)

+-------+-------------------+--------------+------------------------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
old_df = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/workspace/default/detlass")

old_df.show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  1|       C001| Laptop| 50000|
|  2|       C002| Mobile| 15000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
+---+-----------+-------+------+

